# 1.Preparation

Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Change this to your own address.

In [ ]:
!ls "/content/drive/MyDrive/Your own address"

In [ ]:
%cd "/content/drive/MyDrive/Your own address"

In [ ]:
!pip install transformers torch datasets peft wandb
!pip install deepspeed
!pip install rouge-score

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# 2.Data Loading and Formatting

In [ ]:
from datasets import load_from_disk


train_dataset = load_from_disk('/content/drive/MyDrive/Your Trainging data address/')

In [ ]:
def get_unique_ticker_symbols(test_dataset):

    ticker_symbols = set()


    for i in range(len(test_dataset)):
        prompt_content = test_dataset[i]['prompt']


        ticker_symbol = re.search(r"ticker\s([A-Z]+)", prompt_content)


        if ticker_symbol:
            ticker_symbols.add(ticker_symbol.group(1))


    return list(ticker_symbols)



def insert_guidance_after_intro(prompt):


    intro_marker = (
        "[INST]<<SYS>>\n"
        "You are a seasoned stock market analyst. Your task is to list the positive developments and "
        "potential concerns for companies based on relevant news and basic financials from the past weeks, "
        "then provide an analysis and prediction for the companies' stock price movement for the upcoming week."
    )
    guidance_start_marker = "Based on all the information before"
    guidance_end_marker = "Following these instructions, please come up with 2-4 most important positive factors"


    intro_pos = prompt.find(intro_marker)
    guidance_start_pos = prompt.find(guidance_start_marker)
    guidance_end_pos = prompt.find(guidance_end_marker)


    if intro_pos == -1 or guidance_start_pos == -1 or guidance_end_pos == -1:
        return prompt


    guidance_section = prompt[guidance_start_pos:guidance_end_pos].strip()


    new_prompt = (
        f"{prompt[:intro_pos + len(intro_marker)]}\n\n"
        f"{guidance_section}\n\n"
        f"{prompt[intro_pos + len(intro_marker):guidance_start_pos]}"
        f"{prompt[guidance_end_pos:]}"
    )

    return new_prompt


def apply_to_all_prompts_in_dataset(test_dataset):


    updated_dataset = test_dataset.map(lambda x: {"prompt": insert_guidance_after_intro(x["prompt"])})

    return updated_dataset

In [ ]:
train_dataset = apply_to_all_prompts_in_dataset(train_dataset)

Check the trining data

In [ ]:
train_dataset['prompt'][0]

In [ ]:
from huggingface_hub import login

# 输入你的 Hugging Face Access Token
login(token="Use your own key to make the call for the base model")

# 3.Executive training document

In [ ]:
!bash train.sh